# AGILAB worker class paths in Colab

This notebook separates worker-class resolution from the app launch flow.

- `execution_pandas_project` resolves to `ExecutionPandasWorker` and the `PandasWorker` family.
- `flight_project` resolves to `FlightWorker` and the `PolarsWorker` family.
- `uav_relay_queue_project` is shown through `AgiEnv` path resolution, while `DagWorker` is shown separately from the shared core.

It clones the AGILAB repository, installs it in editable mode, and uses a tiny Python 3.12 compatibility shim before importing AGILAB modules.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_worker_paths.ipynb)


In [ ]:
!if [ -d /content/agilab/.git ]; then cd /content/agilab && git pull --ff-only; else rm -rf /content/agilab && git clone --depth 1 https://github.com/ThalesGroup/agilab.git /content/agilab; fi
!python -m pip uninstall -y agilab agi-cluster agi-node agi-env >/dev/null 2>&1 || true
!pip install -q -e /content/agilab/src/agilab/core/agi-env -e /content/agilab/src/agilab/core/agi-node -e /content/agilab/src/agilab/core/agi-cluster -e /content/agilab

import os
import pathlib
from io import UnsupportedOperation as IOUnsupportedOperation
from pathlib import Path
import sys

if not hasattr(pathlib, "UnsupportedOperation"):
    pathlib.UnsupportedOperation = IOUnsupportedOperation

for name in [module for module in list(sys.modules) if module == "agi_cluster" or module.startswith("agi_cluster.") or module == "agi_env" or module.startswith("agi_env.")]:
    del sys.modules[name]
import importlib
importlib.invalidate_caches()

REPO_ROOT = Path("/content/agilab")
CORE_NODE_SRC = REPO_ROOT / "src" / "agilab" / "core" / "agi-node" / "src"
CORE_CLUSTER_SRC = REPO_ROOT / "src" / "agilab" / "core" / "agi-cluster" / "src"
CORE_ENV_SRC = REPO_ROOT / "src" / "agilab" / "core" / "agi-env" / "src"
LOCAL_CORE_PATHS = [REPO_ROOT / "src", CORE_NODE_SRC, CORE_ENV_SRC, CORE_CLUSTER_SRC]
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(path) for path in LOCAL_CORE_PATHS]
    + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else [])
)
for path in LOCAL_CORE_PATHS:
    sys.path.insert(0, str(path))

import types
agi_cluster_pkg = types.ModuleType("agi_cluster")
agi_cluster_pkg.__path__ = [str(CORE_CLUSTER_SRC / "agi_cluster")]
sys.modules["agi_cluster"] = agi_cluster_pkg
import agi_node

BUILTIN_ROOT = REPO_ROOT / "src" / "agilab" / "apps" / "builtin"
for path in [
    BUILTIN_ROOT / "execution_pandas_project" / "src",
    BUILTIN_ROOT / "flight_project" / "src",
]:
    sys.path.insert(0, str(path))

print("Repo root:", REPO_ROOT)
print("Builtin root:", BUILTIN_ROOT)
print("execution_pandas_project source:", (BUILTIN_ROOT / "execution_pandas_project" / "src").is_dir())
print("flight_project source:", (BUILTIN_ROOT / "flight_project" / "src").is_dir())
print("uav_relay_queue_project source:", (BUILTIN_ROOT / "uav_relay_queue_project" / "src").is_dir())


In [ ]:
from agi_env import AgiEnv
from agi_node.dag_worker import DagWorker
from agi_node.pandas_worker import PandasWorker
from agi_node.polars_worker import PolarsWorker

from execution_pandas_worker.execution_pandas_worker import ExecutionPandasWorker
from flight_worker.flight_worker import FlightWorker

def describe_app(app_name: str, imported_cls=None):
    env = AgiEnv(apps_path=BUILTIN_ROOT, app=app_name, verbose=0)
    print(f"\n=== {app_name} ===")
    print("target worker class:", getattr(env, "target_worker_class", None))
    print("worker source:", getattr(env, "worker_path", None))
    print("resolved worker target:", getattr(env, "target_worker", None))
    if imported_cls is not None:
        print("imported class:", imported_cls.__module__ + "." + imported_cls.__name__)
        print("MRO:", " -> ".join(cls.__name__ for cls in imported_cls.mro()[:4]))
    return env

describe_app("execution_pandas_project", ExecutionPandasWorker)
describe_app("flight_project", FlightWorker)
describe_app("uav_relay_queue_project")

print("\n=== shared DagWorker ===")
print("imported class:", DagWorker.__module__ + "." + DagWorker.__name__)
print("MRO:", " -> ".join(cls.__name__ for cls in DagWorker.mro()[:4]))
print("base worker families:", PandasWorker.__name__, "/", PolarsWorker.__name__)


## What this proves

- `agilab` can be installed directly from PyPI in a notebook runtime.
- The built-in app sources can be fetched by cloning the repository inside Colab.
- The notebook includes a small Python 3.12 compatibility shim before importing AGILAB modules.
- `AgiEnv` resolves the worker source path for each built-in app.
- `execution_pandas_project` binds to `ExecutionPandasWorker`, whose base family is `PandasWorker`.
- `flight_project` binds to `FlightWorker`, whose base family is `PolarsWorker`.
- `DagWorker` is available as the shared DAG execution base class.
- `uav_relay_queue_project` stays visible as a resolved app worker path, without pretending it is the same thing as `DagWorker`.

## Next step

After this class-path check works, use the launch notebooks if you want to run the apps end to end.
